# Wind Data Cleaning and Analysis

Code author: Audrey McManemin

Date created: 2024-10-31
Date last modified: 2024-12-13

In [2]:
# Imports
import pandas as pd
import numpy as np
import pathlib

In [14]:
# %% Convert from UTC time to local time in France (UTC+2)

import datetime
from datetime import datetime, timedelta

def convert_datetime_utc_to_local_france(datetime_obj):
    return datetime_obj + timedelta(hours=2)

In [21]:
# import data
wk_dict = {1: [17, 18, 19, 20, 21],
        2: [24, 25, 26, 27, 28],
        3: [9, 10, 11, 12, 13],
        4: [16, 17, 18, 19, 20]
        }

for week, days in wk_dict.items():
    # days = wk_dict[week]
    df_list = []
    
    if week in [1, 2]:
        wk = 6
    elif week in [3, 4]:
        wk = 9

    for d in days:
        if d < 10:
            path = pathlib.PurePath(f'clean_wind_data/Wind_835@Y2024_M0{wk}_D0{d}.CSV')
        else:
            path = pathlib.PurePath(f'clean_wind_data/Wind_835@Y2024_M0{wk}_D{d}.CSV')
        data = pd.read_csv(path, skiprows=1, usecols=range(0, 53))

        og_cols = ["Time and Date", "Status Flags", "Met Air Temp. (C)", "Met Pressure (mbar)", "Met Humidity (%)", "Met Wind Speed (m/s)", "Met Wind Direction (deg)", "Raining", "Fog", "Horizontal Wind Speed (m/s) at 10m", "Wind Direction (deg) at 10m", "Horizontal Wind Speed (m/s) at 50m",  "Horizontal Wind Speed (m/s) at 38m",  "Horizontal Wind Speed (m/s) at 20m", "Wind Direction (deg) at 20m"]
        new_cols = ["datetime_utc", 'status', 'air_temp_C', 'pressure_mbar', 'humidity_pct', 'met_windspeed', 'met_winddir', 'raining', 'fog', 'windspeed_h_10m', 'winddir_10m', 'windspeed_h_50m', 'windspeed_h_38m', 'windspeed_h_20m', 'winddir_20m']
        df = data[og_cols].copy()
        df.columns = new_cols
        
        # filter out errors
        df.replace(9999, np.nan, inplace=True)
        df.replace(9998, np.nan, inplace=True)
        
        # calculate wind shear
        df['windshear_std'] = df[['windspeed_h_10m', 'windspeed_h_20m', 'windspeed_h_38m', 'windspeed_h_50m']].std(axis=1)
        
        df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], dayfirst=True)
        # fix an issue where the month and day are switched for 2024-09-10 to 2024-09-12
        if week == 3 and d >= 10:
            df['datetime_utc'] = pd.to_datetime(f'2024-0{wk}-{d}') + pd.to_timedelta(df['datetime_utc'].dt.strftime('%H:%M:%S'))
        
        # convert datetime in UTC to local (UTC+2)
        df['datetime_local'] = df['datetime_utc'].apply(convert_datetime_utc_to_local_france)

        df_list.append(df)

    wind_data = pd.concat(df_list)
    wind_data.to_csv(f'clean_wind_data/W{week}_wind_data.csv', index=False)

    print(f"Creating W{week}_wind_data.csv")

Creating W1_wind_data.csv
Creating W2_wind_data.csv
Creating W3_wind_data.csv
Creating W4_wind_data.csv
